# 🚀 RAG Document Ingestion Pipeline - GCP Version
## From file_to_ingest Folder → GCS → PostgreSQL with pgvector

**Pipeline:**
1. Scan `file_to_ingest/` folder
2. Upload files to GCS (Google Cloud Storage)
3. Extract text from PDF (dengan page tracking)
4. Chunk text (dengan overlap)
5. Generate embeddings (OpenAI 1536-dim)
6. Insert chunks ke PostgreSQL pgvector
7. Test semantic search

## 0️⃣ Setup - Database Connection & Imports

In [1]:
import os
import sys
import json
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Tuple, Optional
import pandas as pd
import uuid
import pdfplumber
import io
import re
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from sqlalchemy import create_engine, text as sql_text
from sqlalchemy.orm import sessionmaker

# Load .env.local explicitly (for Jupyter/local development)
load_dotenv(Path(".env"))

print("✅ All imports loaded successfully\n")

✅ All imports loaded successfully



## 1️⃣ Setup Database Connection (SQLAlchemy)

In [ ]:
# ============================================================================
# 🔌 DATABASE CONNECTION (SQLAlchemy)
# ============================================================================

DATABASE_URL = os.getenv("DATABASE_URL")
if not DATABASE_URL:
    raise ValueError("DATABASE_URL not set in .env")

# Fix localhost to 127.0.0.1 if needed
if "localhost:5432" in DATABASE_URL:
    DATABASE_URL = DATABASE_URL.replace("localhost:5432", "127.0.0.1:5433")

engine = create_engine(DATABASE_URL, pool_pre_ping=True, pool_size=5, pool_recycle=3600)
SessionLocal = sessionmaker(bind=engine)

print("✅ Database engine created")
print(f"   URL: {DATABASE_URL[:50]}...")

# Test connection
try:
    with engine.connect() as conn:
        result = conn.execute(sql_text("SELECT current_database(), version()"))
        db_name, version = result.fetchone()
        print(f"✅ Connected to: {db_name}")
        print(f"✅ PostgreSQL: {version.split(',')[0]}\n")
except Exception as e:
    print(f"❌ Database connection failed: {e}\n")
    raise

## 2️⃣ Setup GCS Storage Provider

In [ ]:
# ============================================================================
# 📦 GCS STORAGE PROVIDER
# ============================================================================

class GCSStorageProvider:
    """Google Cloud Storage Provider for RAG documents"""
    def __init__(self, bucket_name: str, credentials_path: str = None, project_id: str = None):
        from google.cloud import storage as gcs_storage
        from google.oauth2 import service_account
        
        # Remove gs:// prefix if present
        if bucket_name.startswith('gs://'):
            bucket_name = bucket_name[5:]
        
        self.bucket_name = bucket_name
        
        # Initialize GCS client
        if credentials_path and os.path.exists(credentials_path):
            creds = service_account.Credentials.from_service_account_file(credentials_path)
            self.client = gcs_storage.Client(credentials=creds, project=project_id or creds.project_id)
        else:
            # Use default credentials (Application Default Credentials)
            self.client = gcs_storage.Client(project=project_id)
        
        self.bucket = self.client.bucket(self.bucket_name)
    
    def put(self, file_id: str, content: bytes) -> str:
        """Upload file to GCS"""
        blob = self.bucket.blob(f"uploads/{file_id}.pdf")
        blob.upload_from_string(content, content_type="application/pdf")
        return file_id
    
    def get(self, file_id: str) -> bytes:
        """Download file from GCS"""
        blob = self.bucket.blob(f"uploads/{file_id}.pdf")
        if not blob.exists():
            raise FileNotFoundError(f"File not found in GCS: {file_id}")
        return blob.download_as_bytes()

# Initialize GCS Storage
GCS_BUCKET_NAME = os.getenv("GCS_BUCKET_NAME")
GCS_CREDENTIALS_PATH = os.getenv("GCS_CREDENTIALS_PATH")
GCS_PROJECT_ID = os.getenv("GCS_PROJECT_ID")

if not GCS_BUCKET_NAME:
    raise ValueError("GCS_BUCKET_NAME not set in .env")

try:
    storage = GCSStorageProvider(
        GCS_BUCKET_NAME,
        GCS_CREDENTIALS_PATH,
        GCS_PROJECT_ID
    )
    print(f"✅ GCS Storage initialized")
    print(f"   Bucket: {storage.bucket_name}")
    print(f"   Project: {GCS_PROJECT_ID or 'default'}\n")
except Exception as e:
    print(f"❌ GCS initialization failed: {e}")
    print("   Make sure GCS credentials are configured\n")
    raise

## 3️⃣ Discover Files - Scan file_to_ingest Folder

In [ ]:
# ============================================================================
# 📁 SCAN file_to_ingest FOLDER
# ============================================================================

ingest_folder = Path("file_to_ingest")

# Create folder if not exists
if not ingest_folder.exists():
    print("📂 Folder file_to_ingest belum ada, membuat folder...")
    ingest_folder.mkdir(parents=True)
else:
    print("📂 Folder file_to_ingest sudah ada")

pdf_files = sorted(ingest_folder.glob("*.pdf"))

print("=" * 80)
print(f"📁 Scanning folder: {ingest_folder.absolute()}")
print("=" * 80 + "\n")

if pdf_files:
    print(f"Found {len(pdf_files)} PDF file(s):\n")
    for i, file_path in enumerate(pdf_files, 1):
        file_size = file_path.stat().st_size
        print(f"  [{i}] {file_path.name}")
        print(f"      Size: {file_size:,} bytes\n")
else:
    print("⚠️  No PDF files found in file_to_ingest folder")
    print("\n📝 Please add PDF files to: file_to_ingest/")
    print("   Then run the next cells to process them.\n")

## 4️⃣ Select Files to Process

In [ ]:
# ============================================================================
# 📋 SELECT FILES TO PROCESS
# ============================================================================

# MODIFY THIS: Change which files to process
# Example: [1, 2] to process first and second file
# Example: list(range(1, len(pdf_files) + 1)) to process all
file_indices = list(range(1, len(pdf_files) + 1))  # Process ALL by default

selected_files = []

if pdf_files:
    print("📋 Selected files to ingest:\n")
    for idx in file_indices:
        if 1 <= idx <= len(pdf_files):
            file_path = pdf_files[idx - 1]
            selected_files.append(file_path)
            print(f"  ✅ [{idx}] {file_path.name}")
    
    print(f"\n✅ Total files selected: {len(selected_files)}\n")
else:
    print("❌ No PDF files available to select\n")

## 5️⃣ Upload Files & Create DB Records

In [ ]:
# ============================================================================
# 📤 UPLOAD FILES TO GCS & CREATE DB RECORDS
# ============================================================================

uploaded_documents = []

if selected_files:
    print("=" * 80)
    print(f"Uploading {len(selected_files)} file(s) to GCS")
    print("=" * 80 + "\n")
    
    # Get a valid user_id from database (or use default)
    db = SessionLocal()
    try:
        user_result = db.execute(sql_text("SELECT id FROM \"user\" LIMIT 1")).scalar()
        user_id = user_result if user_result else "00000000-0000-0000-0000-000000000000"
    except:
        user_id = "00000000-0000-0000-0000-000000000000"
    finally:
        db.close()
    
    for file_idx, file_path in enumerate(selected_files, 1):
        try:
            # Read file
            file_content = file_path.read_bytes()
            file_size = len(file_content)
            
            # Generate storage filename
            storage_filename = str(uuid.uuid4())
            
            # Upload to GCS
            storage.put(storage_filename, file_content)
            
            # Create database record
            db_id = str(uuid.uuid4())
            
            db = SessionLocal()
            try:
                db.execute(sql_text("""
                    INSERT INTO document (id, user_id, original_filename, filename, file_path, file_size, status, mime_type)
                    VALUES (:id, :user_id, :original_filename, :filename, :file_path, :file_size, :status, :mime_type)
                """), {
                    "id": db_id,
                    "user_id": user_id,
                    "original_filename": file_path.name,
                    "filename": f"{storage_filename}.pdf",
                    "file_path": f"gs://{storage.bucket_name}/uploads/{storage_filename}.pdf",
                    "file_size": file_size,
                    "status": "UPLOADED",
                    "mime_type": "application/pdf"
                })
                db.commit()
            finally:
                db.close()
            
            uploaded_documents.append({
                "index": file_idx,
                "db_id": db_id,
                "storage_filename": storage_filename,
                "original_filename": file_path.name,
                "file_size": file_size
            })
            
            print(f"  OK [{file_idx}] {file_path.name}")
            print(f"      Size: {file_size:,} bytes")
            print(f"      GCS ID: {storage_filename}")
            print(f"      DB ID: {db_id[:8]}...\n")
        
        except Exception as e:
            print(f"  ERROR [{file_idx}] {file_path.name}: {e}\n")
    
    if uploaded_documents:
        print("=" * 80)
        print(f"SUCCESS: {len(uploaded_documents)}/{len(selected_files)} file(s) uploaded")
        print("=" * 80 + "\n")
    else:
        print("FAILED: No files uploaded\n")
else:
    print("No files selected for upload\n")

## 6️⃣ Extract Text from PDF

In [ ]:
# ============================================================================
# 📄 EXTRACT TEXT FROM PDF (menggunakan pymupdf)
# ============================================================================

import fitz  # pip install pymupdf

def extract_text_pymupdf(pdf_bytes: bytes) -> dict:
    """Extract text from PDF with page tracking"""
    pages_text = {}
    try:
        doc = fitz.open(stream=pdf_bytes, filetype="pdf")
        for i, page in enumerate(doc, 1):
            text = page.get_text("text")
            if text and text.strip():
                pages_text[i] = text
    except Exception as e:
        print(f"Error extracting PDF: {e}")
        raise
    return pages_text

print("✅ PDF extraction function loaded (with page tracking)\n")

# Extract from uploaded documents
extracted_texts = {}  # {db_id: {page_num: text}}

if uploaded_documents:
    print("=" * 80)
    print(f"Extracting text from {len(uploaded_documents)} document(s)")
    print("=" * 80 + "\n")
    
    for doc in uploaded_documents:
        db_id = doc["db_id"]
        storage_filename = doc["storage_filename"]
        filename = doc["original_filename"]
        
        try:
            # Download from GCS
            pdf_bytes = storage.get(storage_filename)
            pages_text = extract_text_pymupdf(pdf_bytes)
            extracted_texts[db_id] = pages_text
            
            total_chars = sum(len(t) for t in pages_text.values())
            print(f"  ✅ {filename}: {len(pages_text)} pages, {total_chars} characters")
        except Exception as e:
            print(f"  ❌ {filename}: {e}")
    
    if extracted_texts:
        print(f"\n✅ Extracted {len(extracted_texts)} documents with page tracking\n")
    else:
        print(f"❌ No documents extracted successfully\n")
else:
    print("⚠️  No documents uploaded yet\n")

## 7️⃣ Execute Text Chunking

In [ ]:
# ============================================================================
# 📦 TEXT CHUNKING (Global chunking with page tracking)
# ============================================================================

def chunk_text_with_pages(
    pages_text: Dict[int, str],
    chunk_size: int = 500,
    overlap: int = 50
) -> List[Tuple[str, int, int]]:
    """
    Chunk text globally (not per-page) while tracking page numbers.
    Returns: List of (chunk_content, start_page, end_page)

    Algorithm:
    1. Combine all pages into one text stream
    2. Split by sentences
    3. Build chunks from sentences (each chunk can span multiple pages)
    4. Track which pages each chunk touches
    """
    chunks_with_pages = []

    # Convert to list of (sentence, page_number) tuples
    all_sentences = []
    for page_num in sorted(pages_text.keys()):
        page_content = pages_text[page_num]
        sentences = re.split(r'(?<=[.!?])\s+', page_content)
        for sentence in sentences:
            if sentence.strip():
                all_sentences.append((sentence, page_num))

    if not all_sentences:
        return chunks_with_pages

    # Build chunks globally
    current_chunk = []
    current_pages = set()
    current_size = 0

    for sentence, page_num in all_sentences:
        words = sentence.split()
        if not words:
            continue

        # If adding this sentence exceeds chunk_size AND we have content, save chunk
        if current_size + len(words) > chunk_size and current_chunk:
            chunk_content = ' '.join(current_chunk)
            start_page = min(current_pages)
            end_page = max(current_pages)
            chunks_with_pages.append((chunk_content, start_page, end_page))

            # OVERLAP: Keep last N words
            overlap_words = current_chunk[-overlap:] if len(current_chunk) > overlap else current_chunk
            current_chunk = overlap_words
            current_size = len(' '.join(current_chunk).split())
            current_pages = {page_num}

        current_chunk.extend(words)
        current_pages.add(page_num)
        current_size += len(words)

    # Save remaining chunk
    if current_chunk:
        chunk_content = ' '.join(current_chunk)
        start_page = min(current_pages)
        end_page = max(current_pages)
        chunks_with_pages.append((chunk_content, start_page, end_page))

    return chunks_with_pages

print("✅ Text chunking function loaded (GLOBAL CHUNKING - spans multiple pages)\n")

# Create chunks
chunks_by_document = {}  # {db_id: [(content, start_page, end_page), ...]}

if extracted_texts:
    print("=" * 80)
    print(f"Creating chunks for {len(extracted_texts)} document(s)")
    print("=" * 80 + "\n")
    
    for doc_id, pages_text in extracted_texts.items():
        chunks_with_pages = chunk_text_with_pages(pages_text, chunk_size=500, overlap=50)
        chunks_by_document[doc_id] = chunks_with_pages
        
        doc = next((d for d in uploaded_documents if d["db_id"] == doc_id), None)
        if doc:
            print(f"  📄 {doc['original_filename']}")
            print(f"     Total Pages: {len(pages_text)}")
            print(f"     Total Chunks: {len(chunks_with_pages)}")
            
            # Analyze chunk distribution
            single_page = sum(1 for _, start, end in chunks_with_pages if start == end)
            multi_page = sum(1 for _, start, end in chunks_with_pages if start != end)
            
            print(f"     Single-page chunks: {single_page}")
            print(f"     Multi-page chunks: {multi_page}")
            
            if chunks_with_pages:
                avg_words = sum(len(c[0].split()) for c in chunks_with_pages) / len(chunks_with_pages)
                print(f"     Avg words/chunk: {avg_words:.0f}\n")
    
    total = sum(len(c) for c in chunks_by_document.values())
    print("=" * 80)
    print(f"SUCCESS: Created {total} chunks")
    print("=" * 80 + "\n")
else:
    print("No text to chunk\n")

## 8️⃣ Load OpenAI & Generate Embeddings

In [ ]:
# ============================================================================
# 🔗 OPENAI EMBEDDINGS
# ============================================================================

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    print("ERROR: OPENAI_API_KEY not set in .env")
    print("Make sure .env has: OPENAI_API_KEY=sk-proj-...")
    raise ValueError("OPENAI_API_KEY not set")

client = OpenAI(api_key=OPENAI_API_KEY)

def generate_embedding(text: str) -> List[float]:
    """Generate 1536-dimensional embedding using OpenAI"""
    response = client.embeddings.create(
        input=text,
        model="text-embedding-3-small"
    )
    return response.data[0].embedding

print("✅ OpenAI embedding function loaded (1536 dimensions)")
print(f"Using API key: {OPENAI_API_KEY[:20]}...\n")

if chunks_by_document:
    print("Testing embedding generation...")
    
    first_doc_id = list(chunks_by_document.keys())[0]
    first_chunk = chunks_by_document[first_doc_id][0][0]
    
    try:
        embedding = generate_embedding(first_chunk[:1000])
        print(f"  OK: Generated {len(embedding)}-dimensional embedding")
        print(f"      First 5 values: {embedding[:5]}\n")
    except Exception as e:
        print(f"  ERROR: {e}\n")
        raise
else:
    print("No chunks for embedding test\n")

## 9️⃣ Embedding Cost Estimation

In [ ]:
# ============================================================================
# 💰 EMBEDDING COST ESTIMATION
# ============================================================================

PRICE_PER_1M_TOKENS = 0.02  # USD per 1M tokens for text-embedding-3-small

print("=" * 80)
print("💰 EMBEDDING COST CONFIGURATION")
print("=" * 80)
print(f"Model: text-embedding-3-small")
print(f"Price: ${PRICE_PER_1M_TOKENS} per 1M input tokens")
print(f"\n✏️  To change price, modify PRICE_PER_1M_TOKENS variable above\n")

def estimate_tokens(text: str) -> int:
    """Estimate token count for text (roughly 1 token per 4 characters)"""
    return len(text) // 4

def estimate_embedding_cost(chunks_data: dict) -> dict:
    """Estimate total cost for embedding all chunks"""
    total_tokens = 0
    total_chunks = 0
    breakdown = []
    
    for doc_id, chunks_with_pages in chunks_data.items():
        doc = next((d for d in uploaded_documents if d["db_id"] == doc_id), None)
        filename = doc["original_filename"] if doc else "Unknown"
        
        doc_tokens = 0
        doc_chunk_count = len(chunks_with_pages)
        
        for chunk_content, _, _ in chunks_with_pages:
            tokens = estimate_tokens(chunk_content)
            doc_tokens += tokens
        
        doc_cost = (doc_tokens / 1_000_000) * PRICE_PER_1M_TOKENS
        breakdown.append((filename, doc_chunk_count, doc_tokens, doc_cost))
        
        total_tokens += doc_tokens
        total_chunks += doc_chunk_count
    
    total_cost = (total_tokens / 1_000_000) * PRICE_PER_1M_TOKENS
    
    return {
        "total_chunks": total_chunks,
        "estimated_tokens": total_tokens,
        "estimated_cost_usd": total_cost,
        "chunks_breakdown": breakdown
    }

print("Cost estimation function loaded\n")

## 🔟 Display Cost Estimate

In [ ]:
# ============================================================================
# 📊 DISPLAY COST ESTIMATE BEFORE EMBEDDING
# ============================================================================

if chunks_by_document:
    print("=" * 80)
    print("📊 EMBEDDING COST ESTIMATION")
    print("=" * 80 + "\n")
    
    # Calculate estimate
    cost_estimate = estimate_embedding_cost(chunks_by_document)
    
    # Show breakdown per document
    print("📄 Cost Breakdown by Document:\n")
    for filename, chunk_count, tokens, cost in cost_estimate["chunks_breakdown"]:
        print(f"  {filename}")
        print(f"    Chunks: {chunk_count}")
        print(f"    Est. Tokens: {tokens:,}")
        print(f"    Est. Cost: ${cost:.6f}\n")
    
    # Show summary
    print("=" * 80)
    print("📋 TOTAL ESTIMATE")
    print("=" * 80)
    print(f"Total Chunks: {cost_estimate['total_chunks']}")
    print(f"Total Est. Tokens: {cost_estimate['estimated_tokens']:,}")
    print(f"Price per 1M Tokens: ${PRICE_PER_1M_TOKENS}")
    print(f"\n💰 TOTAL ESTIMATED COST: ${cost_estimate['estimated_cost_usd']:.6f}")
    print("=" * 80)
    print("\n⚠️  This is an estimate. Actual cost may vary based on OpenAI's tokenization.")
    print("    Run the next cell to proceed with embedding generation.\n")
else:
    print("⚠️  No chunks available for cost estimation\n")

## 1️⃣1️⃣ Insert Chunks to PostgreSQL

In [ ]:
# ============================================================================
# 📤 INSERT CHUNKS TO POSTGRESQL
# ============================================================================

def insert_chunks_to_db(document_id: str, chunks_with_pages: List[Tuple[str, int, int]]):
    """
    Insert chunks with embeddings to PostgreSQL pgvector.
    Supports chunks spanning multiple pages.
    """
    db = SessionLocal()
    try:
        # Update status to PROCESSING
        db.execute(sql_text("""
            UPDATE document SET status = 'PROCESSING' WHERE id = :id
        """), {"id": document_id})
        db.commit()
        print(f"  Status: PROCESSING")
        
        # Insert chunks
        print(f"  Inserting {len(chunks_with_pages)} chunks...")
        
        for idx, (chunk_content, start_page, end_page) in enumerate(chunks_with_pages):
            # Generate embedding
            embedding = generate_embedding(chunk_content)
            
            # Store page range in metadata
            chunk_metadata = {"start_page": start_page, "end_page": end_page}
            
            # Insert chunk
            chunk_id = str(uuid.uuid4())
            db.execute(sql_text("""
                INSERT INTO document_chunk
                (id, document_id, chunk_index, content, page_number, embedding, chunk_metadata, created_at)
                VALUES
                (:id, :document_id, :chunk_index, :content, :page_number, :embedding, :chunk_metadata, :created_at)
            """), {
                "id": chunk_id,
                "document_id": document_id,
                "chunk_index": idx,
                "content": chunk_content,
                "page_number": start_page,
                "embedding": embedding,  # SQLAlchemy handles pgvector conversion
                "chunk_metadata": chunk_metadata,
                "created_at": datetime.utcnow()
            })
            
            db.flush()
            
            if (idx + 1) % 10 == 0:
                print(f"     Progress: {idx + 1}/{len(chunks_with_pages)}")
        
        db.commit()
        
        # Update status to PROCESSED
        db.execute(sql_text("""
            UPDATE document SET status = 'PROCESSED', processed_at = :now WHERE id = :id
        """), {"now": datetime.utcnow(), "id": document_id})
        db.commit()
        print(f"  Status: PROCESSED")
        
    except Exception as e:
        db.rollback()
        db.execute(sql_text("""
            UPDATE document SET status = 'FAILED' WHERE id = :id
        """), {"id": document_id})
        db.commit()
        print(f"  ERROR: {e}")
        raise
    finally:
        db.close()

print("Insert function loaded (handles multi-page chunks)\n")

# Process all documents
if uploaded_documents and chunks_by_document:
    print("=" * 80)
    print(f"Starting ingestion for {len(uploaded_documents)} document(s)")
    print("=" * 80 + "\n")
    
    for doc_idx, doc in enumerate(uploaded_documents, 1):
        document_id = doc["db_id"]
        filename = doc["original_filename"]
        
        if document_id not in chunks_by_document:
            print(f"Skipping {filename} - no chunks\n")
            continue
        
        try:
            chunks_with_pages = chunks_by_document[document_id]
            print(f"[{doc_idx}/{len(uploaded_documents)}] {filename}")
            insert_chunks_to_db(document_id, chunks_with_pages)
            print()
        except Exception as e:
            print(f"FAILED: {e}\n")
    
    print("=" * 80)
    print("INGESTION COMPLETE!")
    print("=" * 80 + "\n")
else:
    print("Missing prerequisites\n")

## 1️⃣2️⃣ Verification - Check Database

In [ ]:
# ============================================================================
# ✅ VERIFICATION - Check Database
# ============================================================================

print("=" * 80)
print("VERIFICATION - Database Contents")
print("=" * 80 + "\n")

db = SessionLocal()
try:
    # Total chunks
    total_chunks = db.execute(sql_text("SELECT COUNT(*) FROM document_chunk")).scalar()
    print(f"Total chunks in database: {total_chunks}\n")
    
    # Document status
    result = db.execute(sql_text("""
        SELECT d.original_filename, d.status, COUNT(dc.id) as chunk_count
        FROM document d LEFT JOIN document_chunk dc ON d.id = dc.document_id
        GROUP BY d.id, d.original_filename, d.status
        ORDER BY d.uploaded_at DESC
    """)).fetchall()
    
    print("Document Status:")
    for filename, status, chunk_count in result:
        print(f"  {filename}")
        print(f"    Status: {status}, Chunks: {chunk_count}\n")
finally:
    db.close()

print("=" * 80 + "\n")

## 1️⃣3️⃣ Semantic Search Test

In [ ]:
# ============================================================================
# 🔍 SEMANTIC SEARCH TEST (using pgvector cosine similarity)
# ============================================================================

def semantic_search(query_text: str, top_k: int = 5) -> list:
    """
    Semantic search menggunakan pgvector cosine similarity (<=>)
    Returns: List of (content, filename, page_number, similarity_score)
    """
    db = SessionLocal()
    try:
        # Generate query embedding
        query_embedding = generate_embedding(query_text)

        # Use pgvector <=> operator for cosine similarity
        result = db.execute(sql_text("""
            SELECT
                dc.content,
                d.original_filename,
                dc.page_number,
                dc.embedding <=> :query_embedding as similarity_score
            FROM document_chunk dc
            JOIN document d ON dc.document_id = d.id
            ORDER BY dc.embedding <=> :query_embedding DESC
            LIMIT :limit
        """), {
            "query_embedding": query_embedding,
            "limit": top_k
        }).fetchall()

        results = []
        for row in result:
            content, filename, page_num, similarity = row
            results.append({
                "content": content,
                "filename": filename,
                "page": page_num,
                "similarity": float(similarity) if similarity else 0.0
            })

        return results
    finally:
        db.close()

print("✅ Semantic search FIXED - using true cosine similarity <=>\n")

# Test search
print("=" * 80)
print("SEMANTIC SEARCH TEST")
print("=" * 80 + "\n")

YOUR_QUERY = "software engineering best practices"

print(f"Query: '{YOUR_QUERY}'\n")

results = semantic_search(YOUR_QUERY, top_k=5)

if results:
    print(f"Found {len(results)} relevant chunks:\n")
    for i, result in enumerate(results, 1):
        print(f"  [{i}] 📄 {result['filename']}")
        print(f"      📍 Page {result['page']}")
        print(f"      ⭐ Similarity: {result['similarity']:.4f}")
        print(f"      📝 {result['content'][:200]}...\n")
else:
    print("❌ No relevant chunks found for this query\n")

print("=" * 80)